## Sentinel Visual Detector: A Deep Learning Approach for Image Classification

This notebook demonstrates how to build a deep learning model using TensorFlow and Keras to classify images from a dataset from kaggle deepfake detection challenge. The model is based on the EfficientNetB0 architecture, which is a powerful convolutional neural network that has been pre-trained on the ImageNet dataset. We will use transfer learning to leverage the pre-trained weights and fine-tune the model for our specific classification task. The notebook includes data augmentation, model training, evaluation, and saving the final model for future use. Let's get started!

In [1]:
from pathlib import Path
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

ModuleNotFoundError: No module named 'tensorflow'

## Setting up constants and paths for the dataset

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 5
SEED = 42

WORKING_DIR = Path("/Users/skakibahammed/code_playground/image_detection/Dataset/")
TRAIN_DIR = WORKING_DIR / "Train"
VAL_DIR = WORKING_DIR / "Validation"
TEST_DIR = WORKING_DIR / "Test"

## Data augmentation and generators for training, validation, and testing

In [ ]:
train_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
    horizontal_flip=True,
    brightness_range=[0.9, 1.1]
)

val_test_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input
)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary"
)

val_generator = val_test_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary"
)

test_generator = val_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

Found 20002 images belonging to 2 classes.
Found 4002 images belonging to 2 classes.
Found 4002 images belonging to 2 classes.


## Building the model using EfficientNetB0 as the base and adding custom layers on top

In [ ]:
base_model = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,050,852 (15.45 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

## Evaluate the model on the test set to see how well it performs on unseen data

In [ ]:
def evaluate_model(model, test_generator):
    loss, acc, auc = model.evaluate(test_generator)
    print(f"Test Accuracy: {acc:.4f}")
    print(f"Test AUC: {auc:.4f}")
    print(f"Test Loss: {loss:.4f}")

# Train the model with early stopping and model checkpointing to save the best model

In [ ]:
callbacks = [
    EarlyStopping(patience=2, restore_best_weights=True),
    ModelCheckpoint("visual_detector.keras", save_best_only=True)
]

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=callbacks
)

Epoch 1/5
626/626 ━━━━━━━━━━━━━━━━━━━━ 286s 451ms/step - accuracy: 0.7577 - auc: 0.8389 - loss: 0.4989 - val_accuracy: 0.7781 - val_auc: 0.9422 - val_loss: 0.4480
Epoch 2/5
626/626 ━━━━━━━━━━━━━━━━━━━━ 292s 467ms/step - accuracy: 0.8012 - auc: 0.8802 - loss: 0.4348 - val_accuracy: 0.8498 - val_auc: 0.9512 - val_loss: 0.3619
Epoch 3/5
626/626 ━━━━━━━━━━━━━━━━━━━━ 292s 466ms/step - accuracy: 0.8075 - auc: 0.8884 - loss: 0.4193 - val_accuracy: 0.7626 - val_auc: 0.9511 - val_loss: 0.4632
Epoch 4/5
626/626 ━━━━━━━━━━━━━━━━━━━━ 274s 438ms/step - accuracy: 0.8115 - auc: 0.8915 - loss: 0.4136 - val_accuracy: 0.7881 - val_auc: 0.9551 - val_loss: 0.4376


In [ ]:
evaluate_model(model, test_generator)

126/126 ━━━━━━━━━━━━━━━━━━━━ 46s 363ms/step - accuracy: 0.7559 - auc: 0.8385 - loss: 0.4927
Test Accuracy: 0.7559
Test AUC: 0.8385
Test Loss: 0.4927


## Fine-tuning the model by unfreezing some of the layers in the base model and training for a few more epochs with a lower learning rate

In [ ]:
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)

history_finetune = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=3
)

Epoch 1/3
626/626 ━━━━━━━━━━━━━━━━━━━━ 336s 529ms/step - accuracy: 0.7740 - auc: 0.8527 - loss: 0.4828 - val_accuracy: 0.8186 - val_auc: 0.9511 - val_loss: 0.3973
Epoch 2/3
626/626 ━━━━━━━━━━━━━━━━━━━━ 328s 524ms/step - accuracy: 0.8343 - auc: 0.9133 - loss: 0.3797 - val_accuracy: 0.8376 - val_auc: 0.9666 - val_loss: 0.3545
Epoch 3/3
626/626 ━━━━━━━━━━━━━━━━━━━━ 324s 518ms/step - accuracy: 0.8583 - auc: 0.9355 - loss: 0.3279 - val_accuracy: 0.8418 - val_auc: 0.9718 - val_loss: 0.3434


In [ ]:
evaluate_model(model, test_generator)

126/126 ━━━━━━━━━━━━━━━━━━━━ 47s 371ms/step - accuracy: 0.8001 - auc: 0.8932 - loss: 0.4127
Test Accuracy: 0.8001
Test AUC: 0.8932
Test Loss: 0.4127


## Saving the final model in Keras format for future use or deployment

In [ ]:
model.save("sentinel_visual_detector.keras")